<a href="https://colab.research.google.com/github/ZeninKris/zmm-movilidad-predictiva/blob/main/02_Feature_Engineering_Tiempo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

# 1. (Una fila por cada hora de 2023 a Dic 2026)
# Esto genera un rango de fechas con frecuencia horaria ('H')
rango_fechas = pd.date_range(start='2023-01-01 00:00:00', end='2026-12-31 23:00:00', freq='h') # lowercase 'h' is standard now

df_tiempo = pd.DataFrame({'fecha_hora': rango_fechas})

# 2. Extraer características temporales básicas (Ciclos de la ciudad)
df_tiempo['año'] = df_tiempo['fecha_hora'].dt.year
df_tiempo['mes'] = df_tiempo['fecha_hora'].dt.month
df_tiempo['dia_semana'] = df_tiempo['fecha_hora'].dt.dayofweek # Lunes=0, Domingo=6
df_tiempo['hora_del_dia'] = df_tiempo['fecha_hora'].dt.hour
df_tiempo['es_fin_de_semana'] = df_tiempo['dia_semana'].apply(lambda x: 1 if x >= 5 else 0)

# 3. Definir Horas Pico (Mañana y Tarde en Monterrey)
# Ejemplo: 7-9 AM y 6-8 PM. Esto es clave para cruzar con tus eventos.
df_tiempo['es_hora_pico'] = df_tiempo['hora_del_dia'].apply(
    lambda h: 1 if (7 <= h <= 9) or (17 <= h <= 20) else 0
)

print(f"Esqueleto creado: {df_tiempo.shape[0]} horas en total.")
print(df_tiempo.head())

Esqueleto creado: 35064 horas en total.
           fecha_hora   año  mes  dia_semana  hora_del_dia  es_fin_de_semana  \
0 2023-01-01 00:00:00  2023    1           6             0                 1   
1 2023-01-01 01:00:00  2023    1           6             1                 1   
2 2023-01-01 02:00:00  2023    1           6             2                 1   
3 2023-01-01 03:00:00  2023    1           6             3                 1   
4 2023-01-01 04:00:00  2023    1           6             4                 1   

   es_hora_pico  
0             0  
1             0  
2             0  
3             0  
4             0  


In [6]:
import pandas as pd
import io

from google.colab import drive
drive.mount('/content/drive')

ruta_eventos = '/content/drive/MyDrive/Proyecto_ZMM/data_raw/eventos_masivos_zmm.csv'

df_eventos = pd.read_csv(ruta_eventos)

# 2. Crear la columna clave: 'fecha_hora_evento'
# Juntamos la fecha y la hora para que coincida con el esqueleto temporal
df_eventos['fecha_hora_evento'] = pd.to_datetime(df_eventos['fecha'] + ' ' + df_eventos['hora_inicio'])

# 3. Eliminar columnas de texto que el modelo no entiende
df_eventos_limpio = df_eventos[['fecha_hora_evento', 'asistencia_estimada', 'nivel_impacto']].copy()

print(df_eventos_limpio.head())

Mounted at /content/drive
    fecha_hora_evento  asistencia_estimada  nivel_impacto
0 2023-01-07 21:10:00                53500              1
1 2023-01-08 19:00:00                42000              1
2 2023-02-11 19:10:00                42000              1
3 2023-03-03 19:00:00                53500              1
4 2023-03-04 19:00:00                42000              1


In [ ]:
# 1. TRUNCAR LOS MINUTOS (Para que 21:10:00 se convierta en 21:00:00)
df_eventos_limpio['fecha_hora_evento'] = df_eventos_limpio['fecha_hora_evento'].dt.floor('h')

# Agrupar por si hay dos eventos a la misma hora (ej. juegan Tigres y hay concierto)
df_eventos_agrupado = df_eventos_limpio.groupby('fecha_hora_evento').agg({
    'asistencia_estimada': 'sum',
    'nivel_impacto': 'max' # Nos quedamos con el evento de mayor impacto
}).reset_index()

# 2. LA FUSIÓN (Left Merge)
df_tiempo = df_tiempo.merge(
    df_eventos_agrupado,
    left_on='fecha_hora',
    right_on='fecha_hora_evento',
    how='left'
)

# 3. LIMPIAR Y CREAR LA ALARMA PREVIA (Variable Adelantada)
df_tiempo['nivel_impacto'] = df_tiempo['nivel_impacto'].fillna(0)
df_tiempo['asistencia_estimada'] = df_tiempo['asistencia_estimada'].fillna(0)
df_tiempo.drop(columns=['fecha_hora_evento'], inplace=True, errors='ignore')

# Si hay evento a las 20:00, el tráfico empieza a las 17:00
df_tiempo['impacto_evento_activo'] = df_tiempo['nivel_impacto'].rolling(window=4, min_periods=1).max().shift(-3)
df_tiempo['impacto_evento_activo'] = df_tiempo['impacto_evento_activo'].fillna(0)

# Verificamos que el cruce funcionó buscando las horas de impacto
print("--- HORAS CON IMPACTO DE EVENTO ---")
print(df_tiempo[df_tiempo['impacto_evento_activo'] > 0].head(10))

# 4. GUARDAR
df_tiempo.to_csv('/content/drive/MyDrive/Proyecto_ZMM/data_processed/esqueleto_tiempo_eventos.csv', index=False)
print("¡Pilar 2 Guardado con Éxito!")